## 2 卷积和池化层
### 2.1 理论计算题
输入图像尺寸：$3 \times 32 \times 32$（通道数 × 高 × 宽）
卷积层参数：卷积核数量 16，卷积核大小 $5 \times 5$，填充 $P=2$，步幅 $S=2$

#### 1. 计算输出特征图尺寸
卷积输出尺寸公式：
$$
H_{out} = \left\lfloor \frac{H_{in} + 2P - K}{S} \right\rfloor + 1
$$
$$
W_{out} = \left\lfloor \frac{W_{in} + 2P - K}{S} \right\rfloor + 1
$$

代入数值计算：
$$
H_{out} = \left\lfloor \frac{32 + 2 \times 2 - 5}{2} \right\rfloor + 1
= \left\lfloor \frac{31}{2} \right\rfloor + 1
= 15 + 1 = 16
$$
$$
W_{out} = \left\lfloor \frac{32 + 2 \times 2 - 5}{2} \right\rfloor + 1 = 16
$$

输出通道数等于卷积核数量，为 $16$。

**输出特征图尺寸**：$\boldsymbol{16 \times 16 \times 16}$

#### 2. 单个输出通道单个像素的乘法次数
$$
\text{乘法次数} = C_{in} \times K_h \times K_w = 3 \times 5 \times 5 = 75
$$
**答案**：75 次

### 2.2 编程题：手动实现二维最大池化前向传播

In [1]:
import numpy as np

def max_pool2d(inputs, kernel_size, stride, padding):
    """
    手动实现2D最大池化前向传播
    :param inputs: 输入张量，shape = (N, C, H, W)
    :param kernel_size: 池化核大小 (kh, kw)
    :param stride: 步幅 (sh, sw)
    :param padding: 填充大小 (ph, pw)
    :return: 池化输出张量
    """
    N, C, H_in, W_in = inputs.shape
    kh, kw = kernel_size
    sh, sw = stride
    ph, pw = padding

    # 对特征图进行填充
    pad_input = np.pad(inputs, ((0,0), (0,0), (ph,ph), (pw,pw)), mode="constant")
    
    # 计算输出特征图高、宽
    H_out = (H_in + 2 * ph - kh) // sh + 1
    W_out = (W_in + 2 * pw - kw) // sw + 1
    
    # 初始化输出
    output = np.zeros((N, C, H_out, W_out))
    
    # 滑动窗口计算最大值
    for n in range(N):
        for c in range(C):
            for h in range(H_out):
                for w in range(W_out):
                    h_start = h * sh
                    h_end = h_start + kh
                    w_start = w * sw
                    w_end = w_start + kw
                    window = pad_input[n, c, h_start:h_end, w_start:w_end]
                    output[n, c, h, w] = np.max(window)
    return output

# 测试代码
if __name__ == "__main__":
    # 构造测试输入: batch=1, channel=1, H=6, W=6
    x = np.random.rand(1, 1, 6, 6)
    pool_out = max_pool2d(x, kernel_size=(2,2), stride=(2,2), padding=(0,0))
    print("输入形状:", x.shape)
    print("最大池化输出形状:", pool_out.shape)
    print("池化结果:\n", pool_out)

输入形状: (1, 1, 6, 6)
最大池化输出形状: (1, 1, 3, 3)
池化结果:
 [[[[0.30536082 0.46041017 0.62137578]
   [0.85853992 0.56145603 0.76040628]
   [0.53398219 0.90896503 0.91854625]]]]


## 3 LeNet, AlexNet, VGG 和 NiN
### 3.1 理论计算题
输入、输出通道数均为 $C$，卷积层**不带偏置**。

#### 1. $5 \times 5$ 卷积层参数量
卷积参数量公式（无偏置）：
$$
\text{Params} = C_{in} \times K_h \times K_w \times C_{out}
$$

$$
\text{Params}_{5\times5} = C \times 5 \times 5 \times C = 25C^2
$$
**答案**：$\boldsymbol{25C^2}$

#### 2. 两层串联 $3 \times 3$ 卷积总参数量
第一层卷积：
$$
\text{Params}_1 = C \times 3 \times 3 \times C = 9C^2
$$
第二层卷积：
$$
\text{Params}_2 = C \times 3 \times 3 \times C = 9C^2
$$

总参数量：
$$
\text{Total Params} = 9C^2 + 9C^2 = 18C^2
$$
**答案**：$\boldsymbol{18C^2}$

### 3.2 编程题：定义 NiN 块

In [2]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.nin_block = nn.Sequential(
            # 第一层普通卷积 + ReLU
            nn.Conv2d(in_channels, out_channels, kernel_size=kernel_size, 
                      stride=stride, padding=padding),
            nn.ReLU(inplace=True),
            # 第一个 1×1 卷积 + ReLU
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            # 第二个 1×1 卷积 + ReLU
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.nin_block(x)

# 测试
if __name__ == "__main__":
    block = NiNBlock(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
    x = torch.randn(1, 3, 32, 32)
    y = block(x)
    print("输入形状:", x.shape)
    print("NiN块输出形状:", y.shape)

输入形状: torch.Size([1, 3, 32, 32])
NiN块输出形状: torch.Size([1, 16, 32, 32])


## 4 Inception, 批量归一化和残差网络
### 4.1 理论计算题：批量归一化计算
已知：
$x_1=2,\ x_2=4,\ x_3=6,\ x_4=8,\ \gamma=2,\ \beta=1,\ \epsilon=0$

批量归一化公式：
$$
\mu = \frac{1}{m}\sum_{i=1}^m x_i
$$
$$
\sigma^2 = \frac{1}{m}\sum_{i=1}^m (x_i-\mu)^2
$$
$$
\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}
$$
$$
y_i = \gamma \cdot \hat{x}_i + \beta
$$

1. 计算均值
$$
\mu = \frac{2+4+6+8}{4} = 5
$$

2. 计算方差
$$
\sigma^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} = 5
$$

3. 归一化结果
$$
\hat{x}_1 = \frac{2-5}{\sqrt{5}} = -\frac{3\sqrt{5}}{5},\quad
\hat{x}_2 = \frac{4-5}{\sqrt{5}} = -\frac{\sqrt{5}}{5}
$$
$$
\hat{x}_3 = \frac{6-5}{\sqrt{5}} = \frac{\sqrt{5}}{5},\quad
\hat{x}_4 = \frac{8-5}{\sqrt{5}} = \frac{3\sqrt{5}}{5}
$$

4. 缩放平移得到最终输出
$$
y_1 = 2 \times \left(-\frac{3\sqrt{5}}{5}\right) + 1 = 1 - \frac{6\sqrt{5}}{5}
$$
$$
y_2 = 2 \times \left(-\frac{\sqrt{5}}{5}\right) + 1 = 1 - \frac{2\sqrt{5}}{5}
$$
$$
y_3 = 2 \times \frac{\sqrt{5}}{5} + 1 = 1 + \frac{2\sqrt{5}}{5}
$$
$$
y_4 = 2 \times \frac{3\sqrt{5}}{5} + 1 = 1 + \frac{6\sqrt{5}}{5}
$$

**最终输出**：
$y_1=1-\dfrac{6\sqrt{5}}{5},\ y_2=1-\dfrac{2\sqrt{5}}{5},\ y_3=1+\dfrac{2\sqrt{5}}{5},\ y_4=1+\dfrac{6\sqrt{5}}{5}$

### 4.2 编程题：自定义残差块 Residual

In [4]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False):
        super(Residual, self).__init__()
        # 主分支：两层3×3卷积 + BN
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # 1×1卷积，用于维度对齐
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        else:
            self.conv3 = None
        
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # 主分支前向
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        # 残差分支维度适配
        if self.conv3 is not None:
            x = self.conv3(x)
        # 残差相加
        y += x
        return self.relu(y)

# 测试
if __name__ == "__main__":
    # 不使用1×1卷积
    res1 = Residual(16, 16, use_1x1conv=False)
    x1 = torch.randn(1, 16, 32, 32)
    print("无1×1卷积输出形状:", res1(x1).shape)

    # 使用1×1卷积改变通道
    res2 = Residual(8, 16, use_1x1conv=True)
    x2 = torch.randn(1, 8, 32, 32)
    print("有1×1卷积输出形状:", res2(x2).shape)

无1×1卷积输出形状: torch.Size([1, 16, 32, 32])
有1×1卷积输出形状: torch.Size([1, 16, 32, 32])


## 5 图像增广, 微调和样式迁移
### 5.1 理论计算题
#### 1. 底层特征层小学习率、顶层输出层大学习率的原因
1. 预训练网络的底层已经学习到边缘、纹理、轮廓等**通用基础视觉特征**，泛化性强，无需大幅更新，因此使用较小学习率或冻结参数，避免破坏已有有效特征。
2. 顶层输出层为新任务随机初始化，参数与目标任务不匹配，需要快速迭代适配新数据集，因此设置较大学习率。

#### 2. 数据集小且与源数据集相似的防过拟合微调策略
1. 冻结全部底层特征提取层，**仅训练最后输出层**，完全复用预训练特征。
2. 若需微调，仅微调少量顶层网络，并使用**极小学习率**。
3. 配合图像增广、权重衰减、Dropout 等正则化方法降低过拟合风险。
4. 不进行全网络参数更新，减少可训练参数量。

### 5.2 编程题：构建图像增广管道

In [6]:
from torchvision import transforms
from PIL import Image

# 图像增广流水线
aug_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(size=224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

# 测试
if __name__ == "__main__":

    img = Image.open("test.png")
    aug_img = aug_pipeline(img)
    print("增广后张量形状:", aug_img.shape)

增广后张量形状: torch.Size([4, 224, 224])


## 6 目标检测,计算机视觉训练技巧
### 6.1 理论计算题：计算 IoU
框格式：$[x_{top}, y_{top}, x_{bottom}, y_{bottom}]$
真实框 $A=[10,10,50,50]$，预测框 $B=[30,30,70,70]$

IoU 公式：
$$
\text{IoU} = \frac{S_{\text{intersection}}}{S_{\text{union}}}
$$

1. 交集坐标
$$
x_1 = \max(10,30) = 30,\quad y_1 = \max(10,30) = 30
$$
$$
x_2 = \min(50,70) = 50,\quad y_2 = \min(50,70) = 50
$$

2. 交集面积
$$
S_{inter} = (50-30) \times (50-30) = 400
$$

3. 单个框面积
$$
S_A = (50-10) \times (50-10) = 1600
$$
$$
S_B = (70-30) \times (70-30) = 1600
$$

4. 并集面积
$$
S_{union} = S_A + S_B - S_{inter} = 1600 + 1600 - 400 = 2800
$$

5. 计算 IoU
$$
\text{IoU} = \frac{400}{2800} = \frac{1}{7}
$$
**答案**：$\boldsymbol{\dfrac{1}{7}}$

### 6.2 编程题：标签平滑交叉熵损失

In [7]:
import torch
import torch.nn as nn

def label_smoothing_ce_loss(logits, labels, num_classes, eps=0.1):
    """
    标签平滑交叉熵损失
    :param logits: 模型原始输出 (N, num_classes)
    :param labels: 真实标签 (N,)
    :param num_classes: 分类总数 K
    :param eps: 平滑因子 ε
    :return: 平均损失
    """
    # 转为概率分布
    log_probs = torch.log_softmax(logits, dim=-1)
    
    # 构造平滑后的标签
    one_hot = torch.zeros_like(log_probs)
    one_hot.scatter_(1, labels.unsqueeze(1), 1.0)
    smooth_label = one_hot * (1 - eps) + (1 - one_hot) * eps / (num_classes - 1)
    
    # 计算损失
    loss = -torch.sum(smooth_label * log_probs, dim=-1)
    return torch.mean(loss)

# 测试
if __name__ == "__main__":
    batch_size = 4
    K = 10
    logits = torch.randn(batch_size, K)
    labels = torch.randint(0, K, (batch_size,))
    loss = label_smoothing_ce_loss(logits, labels, num_classes=K, eps=0.1)
    print("标签平滑交叉熵损失值:", loss.item())

标签平滑交叉熵损失值: 2.7121341228485107
